# 04 - End-to-end field analysis

This notebook demonstrates the M3 end-to-end cycle on one microscope-field-style image:

`field image -> YOLO26n WBC detections -> crops -> DinoBloom-B + MLP 18-class predictions -> uncertainty decisions -> structured JSON`.

It is research-only. The goal is to make each intermediate output visible: bundle metadata, selected image, raw detections, annotated boxes, confidence heatmap, crops, all 18 probabilities, review decisions, timing, and final insights.

## 1. Environment and assets

Install notebook-only visualization dependencies if needed, import the project, and verify required model bundles.

In [ ]:
!pip install -q ultralytics matplotlib pandas seaborn Pillow


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'src/bloodfilm').exists() and (candidate / 'models').exists():
            return candidate
    raise RuntimeError('Could not find repo root containing src/bloodfilm and models')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from matplotlib.patches import Rectangle
from PIL import Image
from ultralytics import YOLO

from bloodfilm.classification.inference import classify_crop
from bloodfilm.classification.bundle import inspect_bundle, load_bundle_documents
from bloodfilm.classification.png import write_rgb_png
from bloodfilm.documents import load_config_document, write_json

DETECTOR_BUNDLE = Path('models/txl-pbc-yolo26n-v0.1')
CLASSIFIER_BUNDLE = Path('models/mll23-dinobloom-b-mlp-v0.1')
BACKBONE_WEIGHTS = Path('models/backbones/dinobloom-b.pth')
OUTPUT_DIR = Path('outputs/e2e')
CROP_DIR = OUTPUT_DIR / 'crops'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CROP_DIR.mkdir(parents=True, exist_ok=True)

for path in [DETECTOR_BUNDLE, CLASSIFIER_BUNDLE, BACKBONE_WEIGHTS]:
    assert path.exists(), f'Missing required asset: {path}'

print('Detector bundle:', DETECTOR_BUNDLE)
print('Classifier bundle:', CLASSIFIER_BUNDLE)
CLASSIFIER_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Backbone weights:', BACKBONE_WEIGHTS)
print('Classifier device:', CLASSIFIER_DEVICE)

In [ ]:
detector_bundle = load_config_document(DETECTOR_BUNDLE / 'bundle.json')
detector_metrics = load_config_document(DETECTOR_BUNDLE / 'metrics.json')
detector_thresholds = load_config_document(DETECTOR_BUNDLE / 'thresholds.json')
classifier_inspection = inspect_bundle(CLASSIFIER_BUNDLE)
classifier_docs = load_bundle_documents(CLASSIFIER_BUNDLE)

display(pd.DataFrame([
    {'asset': 'detector', 'status': 'ok', 'version': detector_bundle['name'], 'recall': detector_metrics['recall'], 'mAP50': detector_metrics['mAP50'], 'mAP50-95': detector_metrics['mAP50_95']},
    {'asset': 'classifier', 'status': classifier_inspection['status'], 'version': classifier_docs['bundle']['name'], 'recall': None, 'mAP50': None, 'mAP50-95': None},
]))
print('Detector thresholds:', json.dumps(detector_thresholds['selected'], indent=2))

## 2. Select one field image

Set `SAMPLE_FIELD=/path/to/image` to use a specific microscope field. If not set, the notebook uses the first TXL-PBC test image so the detector path can be exercised immediately.

In [ ]:
sample_override = os.environ.get('SAMPLE_FIELD')
if sample_override:
    FIELD_IMAGE = Path(sample_override)
else:
    candidates = sorted(Path('data/detection/txl-pbc-wbc-yolo/images/test').glob('*'))
    if not candidates:
        candidates = sorted(Path('data/raw/TXL-PBC/TXL-PBC/images/test').glob('*'))
    assert candidates, 'No sample image found. Run detector prepare-yolo or set SAMPLE_FIELD.'
    FIELD_IMAGE = candidates[0]

image = Image.open(FIELD_IMAGE).convert('RGB')
print('Selected field:', FIELD_IMAGE)
print('Image size:', image.size)
plt.figure(figsize=(7, 7))
plt.imshow(image)
plt.title('Input field image')
plt.axis('off');

## 3. Load detector and classifier

The detector is the packaged TXL-PBC YOLO26n bundle. The classifier path uses the frozen MLL23 DinoBloom-B MLP bundle and emits all 18 probabilities per crop.

In [ ]:
detector_weights = DETECTOR_BUNDLE / detector_bundle['weights_file']
confidence_threshold = float(detector_thresholds['selected']['confidence_threshold'])
iou_threshold = float(detector_thresholds['selected'].get('iou_threshold', 0.5))
detector = YOLO(str(detector_weights))
print('Loaded YOLO detector:', detector_weights)
print('Confidence threshold:', confidence_threshold, 'IoU threshold:', iou_threshold)
print('Classifier version:', classifier_docs['bundle']['name'])

## 4. Detect WBC candidates

Run detection once, retain pixel-space `xyxy` boxes, scores, and per-image latency. Zero detections are reported explicitly and are not interpreted as a zero differential.

In [ ]:
started = time.perf_counter()
results = detector.predict(str(FIELD_IMAGE), conf=confidence_threshold, iou=iou_threshold, imgsz=640, verbose=False)
detector_latency_ms = round((time.perf_counter() - started) * 1000, 3)
result = results[0]
detections = []
for index, box in enumerate(result.boxes):
    x1, y1, x2, y2 = [float(value) for value in box.xyxy[0].detach().cpu().tolist()]
    score = float(box.conf[0].detach().cpu())
    detections.append({'index': index, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'score': score})

detections_df = pd.DataFrame(detections)
print('Detector latency ms:', detector_latency_ms)
print('Detected cells:', len(detections))
display(detections_df)

## 5. Visualize boxes and heatmaps

The first panel shows detector boxes and confidence labels. The second panel is a detector-confidence heatmap created by adding each box score over its covered pixels. This is a visual explanation, not a saliency map.

In [ ]:
image_array = np.array(image)
heatmap = np.zeros(image_array.shape[:2], dtype=float)
for row in detections:
    x1, y1, x2, y2 = [int(round(row[key])) for key in ['x1', 'y1', 'x2', 'y2']]
    heatmap[max(0, y1):min(image_array.shape[0], y2), max(0, x1):min(image_array.shape[1], x2)] += row['score']
if heatmap.max() > 0:
    heatmap = heatmap / heatmap.max()

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].imshow(image_array)
axes[0].set_title('Detected WBC candidate boxes')
for row in detections:
    rect = Rectangle((row['x1'], row['y1']), row['x2'] - row['x1'], row['y2'] - row['y1'], fill=False, color='lime', linewidth=2)
    axes[0].add_patch(rect)
    axes[0].text(row['x1'], max(0, row['y1'] - 4), f"{row['score']:.2f}", color='black', bbox={'facecolor': 'lime', 'alpha': 0.8, 'pad': 1})
axes[0].axis('off')

axes[1].imshow(image_array)
axes[1].imshow(heatmap, cmap='magma', alpha=0.45)
axes[1].set_title('Detector confidence heatmap')
axes[1].axis('off')
plt.tight_layout()
annotated_path = OUTPUT_DIR / 'field_detector_overlay.png'
fig.savefig(annotated_path, dpi=150)
print('Saved overlay:', annotated_path)

## 6. Crop and classify detected cells

Each detector box is expanded by a small padding ratio, saved as a crop, then passed through the MLL23 classifier. Every cell record keeps the detector box, crop box, detector confidence, predicted class, uncertainty decision, and all 18 probabilities.

In [ ]:
def expanded_box(row, width, height, padding_ratio=0.10):
    box_w = row['x2'] - row['x1']
    box_h = row['y2'] - row['y1']
    return {
        'x1': max(0, int(row['x1'] - box_w * padding_ratio)),
        'y1': max(0, int(row['y1'] - box_h * padding_ratio)),
        'x2': min(width, int(row['x2'] + box_w * padding_ratio)),
        'y2': min(height, int(row['y2'] + box_h * padding_ratio)),
    }

cells = []
classification_started = time.perf_counter()
for row in detections:
    crop_box = expanded_box(row, image.width, image.height)
    cell_id = f"{FIELD_IMAGE.stem}_d{row['index']:04d}"
    crop = image.crop((crop_box['x1'], crop_box['y1'], crop_box['x2'], crop_box['y2']))
    crop_path = CROP_DIR / f'{cell_id}.png'
    crop.save(crop_path)
    prediction = classify_crop(bundle_dir=CLASSIFIER_BUNDLE, image_path=crop_path, backbone_weights=BACKBONE_WEIGHTS, device=CLASSIFIER_DEVICE)
    probabilities = prediction['probabilities']
    assert len(probabilities) == 18, 'Expected all 18 probabilities'
    decision = prediction['decision']
    cells.append({
        'cell_id': cell_id,
        'box': {key: round(row[key], 3) for key in ['x1', 'y1', 'x2', 'y2']},
        'crop_box': crop_box,
        'detector_confidence': row['score'],
        'crop_path': str(crop_path),
        'predicted_class': decision.get('label'),
        'classification_confidence': decision.get('confidence'),
        'decision_status': decision.get('status'),
        'uncertainty_reasons': decision.get('uncertainty_reasons', []),
        'probabilities': probabilities,
        'detector_version': detector_bundle['name'],
        'classifier_version': classifier_docs['bundle']['name'],
    })
classification_latency_ms = round((time.perf_counter() - classification_started) * 1000, 3)
print('Classification latency ms:', classification_latency_ms)
print('Cells classified:', len(cells))

In [ ]:
summary_rows = []
for cell in cells:
    summary_rows.append({
        'cell_id': cell['cell_id'],
        'detector_confidence': cell['detector_confidence'],
        'predicted_class': cell['predicted_class'],
        'classification_confidence': cell['classification_confidence'],
        'decision_status': cell['decision_status'],
        'uncertainty_reasons': ', '.join(cell['uncertainty_reasons']),
        'crop_path': cell['crop_path'],
    })
cell_df = pd.DataFrame(summary_rows)
display(cell_df)

if cells:
    probability_df = pd.DataFrame({
        cell['cell_id']: {row['class_code']: row['probability'] for row in cell['probabilities']}
        for cell in cells
    }).T
    display(probability_df)
    plt.figure(figsize=(14, max(3, 0.5 * len(probability_df))))
    sns.heatmap(probability_df, cmap='viridis', vmin=0, vmax=1, annot=True, fmt='.2f')
    plt.title('All 18 probabilities per detected cell')
    plt.xlabel('MLL23 class')
    plt.ylabel('Detected cell')
    probability_heatmap = OUTPUT_DIR / 'classifier_probability_heatmap.png'
    plt.tight_layout()
    plt.savefig(probability_heatmap, dpi=150)
    print('Saved probability heatmap:', probability_heatmap)

## 7. Insights and export

Summarize review burden, zero-detection cases, class distribution, latencies, and saved outputs. The primary JSON artifact is `outputs/e2e/field_analysis.json`, which can be compared across images and future detector/classifier versions.

In [ ]:
status_counts = cell_df['decision_status'].value_counts().to_dict() if not cell_df.empty else {}
class_counts = cell_df['predicted_class'].value_counts().to_dict() if not cell_df.empty else {}
record = {
    'schema_version': 1,
    'research_only': True,
    'status': 'zero_detections' if not cells else 'succeeded',
    'source_image': str(FIELD_IMAGE),
    'image_width': image.width,
    'image_height': image.height,
    'detector_version': detector_bundle['name'],
    'classifier_version': classifier_docs['bundle']['name'],
    'detector_latency_ms': detector_latency_ms,
    'classification_latency_ms': classification_latency_ms,
    'cell_count': len(cells),
    'decision_status_counts': status_counts,
    'predicted_class_counts': class_counts,
    'missed_cell_rate': None,
    'cells': cells,
    'artifacts': {
        'detector_overlay': str(annotated_path),
        'classifier_probability_heatmap': str(OUTPUT_DIR / 'classifier_probability_heatmap.png'),
    },
    'insights': [
        'Zero detections are reported explicitly and are not treated as a zero differential.',
        'Accepted/review/unknown statuses come from the classifier uncertainty policy.',
        'TXL-PBC detector finds WBC candidates; detailed morphology is assigned by the MLL23 classifier.',
    ],
}
output_json = OUTPUT_DIR / 'field_analysis.json'
write_json(output_json, record)
print('Saved structured E2E JSON:', output_json)
print(json.dumps({
    'status': record['status'],
    'cell_count': record['cell_count'],
    'decision_status_counts': status_counts,
    'predicted_class_counts': class_counts,
    'detector_latency_ms': detector_latency_ms,
    'classification_latency_ms': classification_latency_ms,
}, indent=2))